# Quantum Teleportation (Mini)

## Objectives
- Demonstrate state transfer via shared entanglement and classical communication.
- Evaluate average teleportation fidelity over random input states (ideal simulation).

## Setup
```python
import numpy as np
from qiskit import QuantumCircuitr
from qiskit.quantum_info import Statevector, DensityMatrix, random_statevector, state_fidelity, partial_trace
import matplotlib.pyplot as plt
```


## Theory Snapshot
Quantum teleportation transfers an unknown qubit state from Alice to Bob using a shared Bell pair and two classical bits. In an ideal, noiseless setting with perfect feedforward, fidelity approaches 1.0.

## Circuit / Model
We prepare an arbitrary single‑qubit input on qubit 0, entangle qubits 1–2, perform a Bell‑basis measurement on 0–1, and apply Pauli corrections on qubit 2 (emulated here in a unitary fashion).

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, DensityMatrix, random_statevector, state_fidelity, partial_trace
import numpy as np

def teleportation_circuit(alpha, beta, with_corrections=True):
    qc = QuantumCircuit(3)
    # Prepare |psi> = alpha|0> + beta|1> on qubit 0
    qc.initialize([alpha, beta], 0)
    # Create Bell pair on (1,2)
    qc.h(1); qc.cx(1,2)
    # Bell measurement on (0,1)
    qc.cx(0,1); qc.h(0)
    # Classical correction on qubit 2 (post assumed communication of classical bits between Alice and Bob)
    # << crucial step >>
    if with_corrections:
        qc.cx(1, 2)  # X correction
        qc.cz(0, 2)  # Z correction
    return qc

## Experiments
We estimate mean teleportation fidelity over an ensemble of random pure states (Bloch‑sphere uniform).

In [ ]:
def teleport_fidelity_for_state(state, with_corrections=True):
    alpha, beta = state.data
    qc = teleportation_circuit(alpha, beta, with_corrections=with_corrections)
    sv = Statevector.from_label('000').evolve(qc)
    dm = DensityMatrix(sv)
    rho2 = partial_trace(dm, [0,1])
    target = DensityMatrix(state)
    return float(state_fidelity(rho2, target))

vals = [teleport_fidelity_for_state(random_statevector(2), with_corrections=True) for _ in range(50)]
print("Mean fidelity (with corrections):", float(np.mean(vals)))

## Metrics & Plots
- **Mean fidelity** over random inputs (ideal unitary emulation) $\approx 1.0$.

(Optional) Plot sample fidelities.

In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.figure(); plt.plot(vals, marker='o'); plt.ylim(0.0, 1.05)
    plt.xlabel('Trial'); plt.ylabel('Fidelity'); plt.title('Teleportation Fidelity over Random States'); plt.grid(True)
except Exception as e:
    print("Plot skipped:", e)

## Results & Discussion
- Idealized emulation yields near‑unit fidelity; explicit measurement + conditional corrections gives identical results on a noiseless simulator.
- On hardware or with noise models, fidelity decreases due to gate and measurement errors; feedforward latency and decoherence also matter.

## References
- Bennett et al., *Teleporting an Unknown Quantum State*, PRL (1993)
- IBM Qiskit Textbook — *Teleportation*
